# <font color="#418FDE" size="6.5" uppercase>**Structured Outputs**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Distinguish multiclass, multilabel, multioutput classification, and multioutput regression tasks. 
- Apply wrapper strategies such as one-vs-rest, one-vs-one, and output-code classification. 
- Use classifier and regressor chains to model dependencies among multiple outputs. 


## **1. Output Types**

### **1.1. Output Type Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_01_01.jpg?v=1783851187" width="250">



>* Prediction tasks can have different output structures.
>* Output type shapes data, metrics, models.

>* Classify by output count and type.
>* Output structure shapes assumptions and modeling.

>* Multilabel uses many labels from one set.
>* Output structure guides models and evaluation.



In [ ]:
#@title Python Code - Output Type Basics

# This script compares common structured output types.
# Small examples make target shapes easy.
# No model training is needed here.

import numpy as np
import pandas as pd

# Set a deterministic seed.
rng = np.random.default_rng(7)

# Build one multiclass target.
multiclass = np.array(["cat", "dog", "horse", "dog"])

# Build one multilabel target matrix.
multilabel = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 0], [0, 0, 1]])

# Build one multioutput classification table.
multioutput_class = pd.DataFrame({"size": ["S", "M", "L", "M"], "priority": ["low", "high", "medium", "low"]})

# Build one multioutput regression table.
multioutput_reg = pd.DataFrame({"temp_c": [20.5, 21.0, 19.8, 22.1], "wind_kph": [12.0, 8.5, 15.2, 10.1]})

# Check matching sample counts.
counts = [len(multiclass), len(multilabel), len(multioutput_class), len(multioutput_reg)]

# Validate all examples align.
assert len(set(counts)) == 1, "Sample counts must match."

# Summarize each output type.
print("Multiclass: one categorical answer per sample.")
print("Example shape:", multiclass.shape, "Sample:", multiclass[0])
print("Multilabel: many labels from one shared label set.")
print("Example shape:", multilabel.shape, "Sample:", multilabel[0].tolist())

# Continue the summary.
print("Multioutput classification: several categorical targets together.")
print("Columns:", multioutput_class.columns.tolist(), "Row:", multioutput_class.iloc[0].to_dict())
print("Multioutput regression: several numeric targets together.")
print("Columns:", multioutput_reg.columns.tolist(), "Row:", multioutput_reg.iloc[0].to_dict())



### **1.2. Multilabel Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_01_02.jpg?v=1783851189" width="250">



>* One input can have multiple labels.
>* Labels are separate yes or no decisions.

>* Outputs are combinations, not single classes.
>* Missing labels mean properties are absent.

>* Labels can depend on each other.
>* Good models predict realistic label combinations.



In [ ]:
#@title Python Code - Multilabel Classification

# This script demonstrates simple multilabel targets.
# One item can have several labels.
# Outputs are multiple yes or no values.

import numpy as np
import pandas as pd

# Fix randomness for reproducible teaching output.
rng = np.random.default_rng(7)

# Create tiny item names and features.
items = ["Email", "Photo", "Song", "Article"]
features = pd.DataFrame({
    "length_score": [2, 5, 3, 4],
    "topic_score": [5, 2, 1, 4]
})

# Build multilabel targets for each item.
labels = pd.DataFrame({
    "important": [1, 0, 0, 1],
    "work_related": [1, 0, 0, 1],
    "contains_people": [0, 1, 0, 0]
})

# Check that rows align safely.
assert len(items) == len(features) == len(labels)

# Combine names, features, and labels.
example_table = pd.concat([
    pd.Series(items, name="item"),
    features,
    labels
], axis=1)

# Show a compact beginner friendly table.
print("Multilabel example table:")
print(example_table.to_string(index=False))

# Count labels per item.
label_counts = labels.sum(axis=1).to_numpy()

# Explain why this is multilabel.
print("\nLabels per item:", label_counts.tolist())
print("A row can have several ones.")
print("That means labels can coexist.")
print("This is not multiclass prediction.")



### **1.3. Multilabel Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_01_03.jpg?v=1783851192" width="250">



>* One example can have multiple labels.
>* Unlike multiclass, labels coexist, not compete.

>* Real examples often need multiple labels.
>* Prediction means choosing the correct label set.

>* Labels differ in frequency and co-occur.
>* Evaluation balances finding labels and avoiding extras.



In [ ]:
#@title Python Code - Multilabel Classification

# This script demonstrates multilabel targets simply.
# One item can have several labels.
# Outputs are sets, not single classes.

import numpy as np
import pandas as pd

# Build tiny examples with overlapping labels.
items = ["Article A", "Photo B", "Song C", "Post D"]
labels = ["politics", "economics", "people", "sunset"]
Y = np.array([[1, 1, 0, 0], [0, 0, 1, 1], [0, 1, 0, 0], [0, 0, 1, 0]])

# Check that dimensions match safely.
assert Y.shape == (len(items), len(labels))
df = pd.DataFrame(Y, index=items, columns=labels)

# Count labels assigned to each item.
label_counts = df.sum(axis=1)
multilabel_flags = label_counts > 1

# Show the multilabel target table.
print("Multilabel target matrix:")
print(df)
print()
print("Labels per item:", label_counts.to_dict())

# Explain why this is multilabel.
print("Items with multiple labels:", multilabel_flags.to_dict())
print("Example output for Photo B:", df.loc["Photo B"].to_dict())
print("This is multilabel because one row can contain several ones.")



## **2. Multiclass Wrappers**

### **2.1. Wrapper Strategy Overview**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_02_01.jpg?v=1783851175" width="250">



>* Wrappers adapt binary models for multiclass tasks.
>* They bridge real categories without changing algorithms.

>* Wrappers split multiclass tasks into binary problems.
>* Different comparison schemes combine simpler decisions.

>* Wrappers trade off speed, cost, and balance.
>* Choose methods to fit data and needs.



In [ ]:
#@title Python Code - Wrapper Strategy Overview

# Wrapper methods split multiclass tasks simply.
# This overview compares three common strategies.
# Tiny examples keep the idea concrete.

# Store class names for examples.
classes = ["cat", "dog", "bird"]

# Count binary models for one-vs-rest.
ovr_models = len(classes)

# Count pairwise models for one-vs-one.
ovo_models = len(classes) * (len(classes) - 1) // 2

# Create a tiny output codebook.
codebook = {
    "cat":  [1, 0, 1],
    "dog":  [0, 1, 1],
    "bird": [1, 1, 0],
}

# Simulate binary outputs from classifiers.
predicted_code = [1, 0, 0]

# Measure code mismatch using Hamming distance.
def distance(code_a, code_b):
    return sum(a != b for a, b in zip(code_a, code_b))

# Decode by choosing the nearest class.
best_class = min(
    codebook,
    key=lambda name: distance(codebook[name], predicted_code)
)

# Print a compact strategy summary.
print("Classes:", classes)
print("One-vs-rest models:", ovr_models)
print("One-vs-one models:", ovo_models)
print("Output code prediction:", predicted_code)
print("Decoded class:", best_class)
print("Key idea: wrappers reuse binary classifiers.")



### **2.2. Wrapper Strategy Overview**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_02_02.jpg?v=1783851177" width="250">



>* Wrappers turn multiclass tasks into binary decisions.
>* They let binary classifiers handle many classes.

>* Wrappers organize binary models for multiclass tasks.
>* Strategy choice changes cost, speed, and robustness.

>* Wrappers shape errors, imbalance, and confidence.
>* Best choice depends on data and classes.



### **2.3. One Versus Rest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_02_03.jpg?v=1783851179" width="250">



>* Train one binary model per class.
>* Choose the class with highest score.

>* Practical, scalable, reusable for many classes.
>* Independent training risks imbalance and weaker boundaries.

>* Comparable scores are crucial for fair predictions.
>* OvR stays simple, flexible, and practical.



## **3. Classifier Regressor Chains**

### **3.1. Chain Dependencies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_03_01.jpg?v=1783851181" width="250">



>* Earlier outputs help predict later outputs.
>* Chains capture label patterns beyond features.

>* Chains learn links between related outputs.
>* Early errors and chain order affect results.

>* Chains model linked outputs step by step.
>* They capture dependencies, not true causation.



### **3.2. Modeling Output Dependencies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_03_02.jpg?v=1783851185" width="250">



>* Outputs often depend on each other.
>* Chains use earlier predictions for context.

>* Earlier predictions guide later output predictions.
>* Chains reveal hidden relationships among outputs.

>* Early chain errors can affect later predictions.
>* Good output order captures useful dependencies.



In [ ]:
#@title Python Code - Modeling Output Dependencies

# This script shows dependent output prediction.
# We compare isolated and chained models.
# Small synthetic data keeps ideas clear.

import numpy as np
from sklearn.datasets import make_multilabel_classification
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import ClassifierChain, MultiOutputClassifier

# Set a deterministic random seed.
rng = np.random.RandomState(7)

# Create features with correlated labels.
X, Y = make_multilabel_classification(
    n_samples=240, n_features=6, n_classes=3,
    n_labels=2, allow_unlabeled=False, random_state=7
)

# Strengthen dependency between later outputs.
Y[:, 1] = ((Y[:, 0] + Y[:, 1]) > 0).astype(int)
Y[:, 2] = ((Y[:, 1] + Y[:, 2]) > 0).astype(int)

# Split into train and test parts.
X_train, X_test = X[:180], X[180:]
Y_train, Y_test = Y[:180], Y[180:]

# Validate shapes before fitting models.
assert X_train.shape[0] == Y_train.shape[0]
assert Y_train.shape[1] == 3

# Fit independent models per output.
base = LogisticRegression(max_iter=300, solver="lbfgs")
independent = MultiOutputClassifier(base)
independent.fit(X_train, Y_train)

# Fit a classifier chain model.
chain = ClassifierChain(base, order=[0, 1, 2])
chain.fit(X_train, Y_train)

# Predict with both approaches.
pred_ind = independent.predict(X_test)
pred_chain = chain.predict(X_test)

# Compute exact match accuracy.
acc_ind = np.mean(np.all(pred_ind == Y_test, axis=1))
acc_chain = np.mean(np.all(pred_chain == Y_test, axis=1))

# Show a few readable examples.
print("Independent exact match:", round(float(acc_ind), 3))
print("Chain exact match:", round(float(acc_chain), 3))
print("First true labels:", Y_test[0].tolist())
print("Independent prediction:", pred_ind[0].tolist())
print("Chain prediction:", pred_chain[0].tolist())
print("Dependency idea: later labels use earlier predictions.")



### **3.3. Regressor Chains**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_A/image_03_03.jpg?v=1783851183" width="250">



>* Regressor chains predict related outputs sequentially.
>* Earlier predictions help later targets use dependencies.

>* Chains exploit dependencies among continuous outputs.
>* They balance separate and joint modeling.

>* Chain order matters; errors can spread.
>* Shared outputs can improve realistic predictions.



In [ ]:
#@title Python Code - Regressor Chains

# Regressor chains model dependent numeric targets.
# This example compares independent and chained predictions.
# Small synthetic data keeps the lesson simple.

import numpy as np

# Build a tiny deterministic dataset.
rng = np.random.default_rng(7)
X = rng.normal(size=(180, 2))

y1 = 3 * X[:, 0] - X[:, 1]
y2 = 2 * y1 + 0.5 * X[:, 1]
Y = np.column_stack((y1, y2))

# Split data into train and test parts.
X_train, X_test = X[:140], X[140:]
Y_train, Y_test = Y[:140], Y[140:]

# Check shapes before fitting models.
assert X_train.shape[0] == Y_train.shape[0]
assert Y_train.shape[1] == 2

# Fit two independent linear regressions.
X1 = np.column_stack((np.ones(len(X_train)), X_train))
B1 = np.linalg.lstsq(X1, Y_train[:, 0], rcond=None)[0]
B2 = np.linalg.lstsq(X1, Y_train[:, 1], rcond=None)[0]

# Predict both targets independently.
X1_test = np.column_stack((np.ones(len(X_test)), X_test))
pred1_ind = X1_test @ B1
pred2_ind = X1_test @ B2

# Fit a simple regressor chain.
X2_chain = np.column_stack((np.ones(len(X_train)), X_train, Y_train[:, 0]))
B2_chain = np.linalg.lstsq(X2_chain, Y_train[:, 1], rcond=None)[0]

# Predict first target, then second target.
pred1_chain = pred1_ind.copy()
X2_test_chain = np.column_stack((np.ones(len(X_test)), X_test, pred1_chain))
pred2_chain = X2_test_chain @ B2_chain

# Compute mean squared errors.
mse1_ind = np.mean((Y_test[:, 0] - pred1_ind) ** 2)
mse2_ind = np.mean((Y_test[:, 1] - pred2_ind) ** 2)

# Compute chained second-target error.
mse2_chain = np.mean((Y_test[:, 1] - pred2_chain) ** 2)
summary = [
    f"Test samples: {len(X_test)}",
    f"Independent target 1 MSE: {mse1_ind:.6f}",
    f"Independent target 2 MSE: {mse2_ind:.6f}",
    f"Chained target 2 MSE: {mse2_chain:.6f}",
]

# Explain the result briefly.
summary.append("Lower chained target 2 error shows useful dependency.")
print("\n".join(summary))



# <font color="#418FDE" size="6.5" uppercase>**Structured Outputs**</font>


In this lecture, you learned to:
- Distinguish multiclass, multilabel, multioutput classification, and multioutput regression tasks. 
- Apply wrapper strategies such as one-vs-rest, one-vs-one, and output-code classification. 
- Use classifier and regressor chains to model dependencies among multiple outputs. 

In the next Lecture (Lecture B), we will go over 'Semi Supervised'